In [ ]:
import builtins

original_print = print
def print(*args, **kwargs):
    with open("TripleSearch30.txt", "a") as f:
        f.write(" ".join(str(a) for a in args) + "\n")
    original_print(*args, **kwargs)


In [2]:
import numpy as np
from scipy.io import wavfile
from scipy.linalg import toeplitz, norm
from scipy.fftpack import fft
import math
from scipy import signal
from pesq import pesq

""" 
This is a python script which can be regarded as implementation of matlab script "compute_metrics.m".

Usage: 
    pesq, csig, cbak, covl, ssnr, stoi = compute_metrics(cleanFile, enhancedFile, Fs, path)
    cleanFile: clean audio as array or path if path is equal to 1
    enhancedFile: enhanced audio as array or path if path is equal to 1
    Fs: sampling rate, usually equals to 8000 or 16000 Hz
    path: whether the "cleanFile" and "enhancedFile" arguments are in .wav format or in numpy array format, 
          1 indicates "in .wav format"
          
Example call:
    pesq_output, csig_output, cbak_output, covl_output, ssnr_output, stoi_output = \
            compute_metrics(target_audio, output_audio, 16000, 0)
"""


def compute_metrics(cleanFile, enhancedFile, Fs, path):
    alpha = 0.95

    
    data1 = cleanFile
    data2 = enhancedFile
    sampling_rate1 = Fs
    sampling_rate2 = Fs

        
    if len(data1) != len(data2):
        length = min(len(data1), len(data2))
        data1 = data1[0:length] + np.spacing(1)
        data2 = data2[0:length] + np.spacing(1)

    # compute the WSS measure
    wss_dist_vec = wss(data1, data2, sampling_rate1)
    wss_dist_vec = np.sort(wss_dist_vec)
    wss_dist = np.mean(wss_dist_vec[0 : round(np.size(wss_dist_vec) * alpha)])

    # compute the LLR measure
    LLR_dist = llr(data1, data2, sampling_rate1)
    LLRs = np.sort(LLR_dist)
    LLR_len = round(np.size(LLR_dist) * alpha)
    llr_mean = np.mean(LLRs[0:LLR_len])

    # compute the SNRseg
    snr_dist, segsnr_dist = snr(data1, data2, sampling_rate1)
    snr_mean = snr_dist
    segSNR = np.mean(segsnr_dist)

    # compute the pesq
    pesq_mos = pesq(sampling_rate1, data1, data2, "wb")

    # now compute the composite measures
    CSIG = 3.093 - 1.029 * llr_mean + 0.603 * pesq_mos - 0.009 * wss_dist
    CSIG = max(1, CSIG)
    CSIG = min(5, CSIG)  # limit values to [1, 5]
    CBAK = 1.634 + 0.478 * pesq_mos - 0.007 * wss_dist + 0.063 * segSNR
    CBAK = max(1, CBAK)
    CBAK = min(5, CBAK)  # limit values to [1, 5]
    COVL = 1.594 + 0.805 * pesq_mos - 0.512 * llr_mean - 0.007 * wss_dist
    COVL = max(1, COVL)
    COVL = min(5, COVL)  # limit values to [1, 5]

    STOI = stoi(data1, data2, sampling_rate1)

    return pesq_mos, CSIG, CBAK, COVL, segSNR, STOI


def wss(clean_speech, processed_speech, sample_rate):
    # Check the length of the clean and processed speech, which must be the same.
    clean_length = np.size(clean_speech)
    processed_length = np.size(processed_speech)
    if clean_length != processed_length:
        raise ValueError("Files must have same length.")

    # Global variables
    winlength = (np.round(30 * sample_rate / 1000)).astype(
        int
    )  # window length in samples
    skiprate = (np.floor(np.divide(winlength, 4))).astype(int)  # window skip in samples
    max_freq = (np.divide(sample_rate, 2)).astype(int)  # maximum bandwidth
    num_crit = 25  # number of critical bands

    USE_FFT_SPECTRUM = 1  # defaults to 10th order LP spectrum
    n_fft = (np.power(2, np.ceil(np.log2(2 * winlength)))).astype(int)
    n_fftby2 = (np.multiply(0.5, n_fft)).astype(int)  # FFT size/2
    Kmax = 20.0  # value suggested by Klatt, pg 1280
    Klocmax = 1.0  # value suggested by Klatt, pg 1280

    # Critical Band Filter Definitions (Center Frequency and Bandwidths in Hz)
    cent_freq = np.array(
        [
            50.0000,
            120.000,
            190.000,
            260.000,
            330.000,
            400.000,
            470.000,
            540.000,
            617.372,
            703.378,
            798.717,
            904.128,
            1020.38,
            1148.30,
            1288.72,
            1442.54,
            1610.70,
            1794.16,
            1993.93,
            2211.08,
            2446.71,
            2701.97,
            2978.04,
            3276.17,
            3597.63,
        ]
    )
    bandwidth = np.array(
        [
            70.0000,
            70.0000,
            70.0000,
            70.0000,
            70.0000,
            70.0000,
            70.0000,
            77.3724,
            86.0056,
            95.3398,
            105.411,
            116.256,
            127.914,
            140.423,
            153.823,
            168.154,
            183.457,
            199.776,
            217.153,
            235.631,
            255.255,
            276.072,
            298.126,
            321.465,
            346.136,
        ]
    )

    bw_min = bandwidth[0]  # minimum critical bandwidth

    # Set up the critical band filters.
    # Note here that Gaussianly shaped filters are used.
    # Also, the sum of the filter weights are equivalent for each critical band filter.
    # Filter less than -30 dB and set to zero.
    min_factor = math.exp(-30.0 / (2.0 * 2.303))  # -30 dB point of filter
    crit_filter = np.empty((num_crit, n_fftby2))
    for i in range(num_crit):
        f0 = (cent_freq[i] / max_freq) * n_fftby2
        bw = (bandwidth[i] / max_freq) * n_fftby2
        norm_factor = np.log(bw_min) - np.log(bandwidth[i])
        j = np.arange(n_fftby2)
        crit_filter[i, :] = np.exp(
            -11 * np.square(np.divide(j - np.floor(f0), bw)) + norm_factor
        )
        cond = np.greater(crit_filter[i, :], min_factor)
        crit_filter[i, :] = np.where(cond, crit_filter[i, :], 0)
    # For each frame of input speech, calculate the Weighted Spectral Slope Measure
    num_frames = int(
        clean_length / skiprate - (winlength / skiprate)
    )  # number of frames
    start = 0  # starting sample
    window = 0.5 * (
        1 - np.cos(2 * math.pi * np.arange(1, winlength + 1) / (winlength + 1))
    )

    distortion = np.empty(num_frames)
    for frame_count in range(num_frames):
        # (1) Get the Frames for the test and reference speech. Multiply by Hanning Window.
        clean_frame = clean_speech[start : start + winlength] / 32768
        processed_frame = processed_speech[start : start + winlength] / 32768
        clean_frame = np.multiply(clean_frame, window)
        processed_frame = np.multiply(processed_frame, window)
        # (2) Compute the Power Spectrum of Clean and Processed
        # if USE_FFT_SPECTRUM:
        clean_spec = np.square(np.abs(fft(clean_frame, n_fft)))
        processed_spec = np.square(np.abs(fft(processed_frame, n_fft)))

        # (3) Compute Filterbank Output Energies (in dB scale)
        clean_energy = np.matmul(crit_filter, clean_spec[0:n_fftby2])
        processed_energy = np.matmul(crit_filter, processed_spec[0:n_fftby2])

        clean_energy = 10 * np.log10(np.maximum(clean_energy, 1e-10))
        processed_energy = 10 * np.log10(np.maximum(processed_energy, 1e-10))

        # (4) Compute Spectral Slope (dB[i+1]-dB[i])
        clean_slope = clean_energy[1:num_crit] - clean_energy[0 : num_crit - 1]
        processed_slope = (
            processed_energy[1:num_crit] - processed_energy[0 : num_crit - 1]
        )

        # (5) Find the nearest peak locations in the spectra to each critical band.
        #     If the slope is negative, we search to the left. If positive, we search to the right.
        clean_loc_peak = np.empty(num_crit - 1)
        processed_loc_peak = np.empty(num_crit - 1)

        for i in range(num_crit - 1):
            # find the peaks in the clean speech signal
            if clean_slope[i] > 0:  # search to the right
                n = i
                while (n < num_crit - 1) and (clean_slope[n] > 0):
                    n = n + 1
                clean_loc_peak[i] = clean_energy[n - 1]
            else:  # search to the left
                n = i
                while (n >= 0) and (clean_slope[n] <= 0):
                    n = n - 1
                clean_loc_peak[i] = clean_energy[n + 1]

            # find the peaks in the processed speech signal
            if processed_slope[i] > 0:  # search to the right
                n = i
                while (n < num_crit - 1) and (processed_slope[n] > 0):
                    n = n + 1
                processed_loc_peak[i] = processed_energy[n - 1]
            else:  # search to the left
                n = i
                while (n >= 0) and (processed_slope[n] <= 0):
                    n = n - 1
                processed_loc_peak[i] = processed_energy[n + 1]

        # (6) Compute the WSS Measure for this frame. This includes determination of the weighting function.
        dBMax_clean = np.max(clean_energy)
        dBMax_processed = np.max(processed_energy)
        """
        The weights are calculated by averaging individual weighting factors from the clean and processed frame.
        These weights W_clean and W_processed should range from 0 to 1 and place more emphasis on spectral peaks
        and less emphasis on slope differences in spectral valleys.
        This procedure is described on page 1280 of Klatt's 1982 ICASSP paper.
        """
        Wmax_clean = np.divide(
            Kmax, Kmax + dBMax_clean - clean_energy[0 : num_crit - 1]
        )
        Wlocmax_clean = np.divide(
            Klocmax, Klocmax + clean_loc_peak - clean_energy[0 : num_crit - 1]
        )
        W_clean = np.multiply(Wmax_clean, Wlocmax_clean)

        Wmax_processed = np.divide(
            Kmax, Kmax + dBMax_processed - processed_energy[0 : num_crit - 1]
        )
        Wlocmax_processed = np.divide(
            Klocmax, Klocmax + processed_loc_peak - processed_energy[0 : num_crit - 1]
        )
        W_processed = np.multiply(Wmax_processed, Wlocmax_processed)

        W = np.divide(np.add(W_clean, W_processed), 2.0)
        slope_diff = np.subtract(clean_slope, processed_slope)[0 : num_crit - 1]
        distortion[frame_count] = np.dot(W, np.square(slope_diff)) / np.sum(W)
        # this normalization is not part of Klatt's paper, but helps to normalize the measure.
        # Here we scale the measure by the sum of the weights.
        start = start + skiprate
    return distortion


def llr(clean_speech, processed_speech, sample_rate):
    # Check the length of the clean and processed speech.  Must be the same.
    clean_length = np.size(clean_speech)
    processed_length = np.size(processed_speech)
    if clean_length != processed_length:
        raise ValueError("Both Speech Files must be same length.")

    # Global Variables
    winlength = (np.round(30 * sample_rate / 1000)).astype(
        int
    )  # window length in samples
    skiprate = (np.floor(winlength / 4)).astype(int)  # window skip in samples
    if sample_rate < 10000:
        P = 10  # LPC Analysis Order
    else:
        P = 16  # this could vary depending on sampling frequency.

    # For each frame of input speech, calculate the Log Likelihood Ratio
    num_frames = int((clean_length - winlength) / skiprate)  # number of frames
    start = 0  # starting sample
    window = 0.5 * (
        1 - np.cos(2 * math.pi * np.arange(1, winlength + 1) / (winlength + 1))
    )

    distortion = np.empty(num_frames)
    for frame_count in range(num_frames):
        # (1) Get the Frames for the test and reference speech. Multiply by Hanning Window.
        clean_frame = clean_speech[start : start + winlength]
        processed_frame = processed_speech[start : start + winlength]
        clean_frame = np.multiply(clean_frame, window)
        processed_frame = np.multiply(processed_frame, window)

        # (2) Get the autocorrelation lags and LPC parameters used to compute the LLR measure.
        R_clean, Ref_clean, A_clean = lpcoeff(clean_frame, P)
        R_processed, Ref_processed, A_processed = lpcoeff(processed_frame, P)

        # (3) Compute the LLR measure
        numerator = np.dot(np.matmul(A_processed, toeplitz(R_clean)), A_processed)
        denominator = np.dot(np.matmul(A_clean, toeplitz(R_clean)), A_clean)
        distortion[frame_count] = math.log(numerator / denominator)
        start = start + skiprate
    return distortion


def lpcoeff(speech_frame, model_order):
    # (1) Compute Autocorrelation Lags
    winlength = np.size(speech_frame)
    R = np.empty(model_order + 1)
    E = np.empty(model_order + 1)
    for k in range(model_order + 1):
        R[k] = np.dot(speech_frame[0 : winlength - k], speech_frame[k:winlength])

    # (2) Levinson-Durbin
    a = np.ones(model_order)
    a_past = np.empty(model_order)
    rcoeff = np.empty(model_order)
    E[0] = R[0]
    for i in range(model_order):
        a_past[0:i] = a[0:i]
        sum_term = np.dot(a_past[0:i], R[i:0:-1])
        rcoeff[i] = (R[i + 1] - sum_term) / E[i]
        a[i] = rcoeff[i]
        if i == 0:
            a[0:i] = a_past[0:i] - np.multiply(a_past[i - 1 : -1 : -1], rcoeff[i])
        else:
            a[0:i] = a_past[0:i] - np.multiply(a_past[i - 1 :: -1], rcoeff[i])
        E[i + 1] = (1 - rcoeff[i] * rcoeff[i]) * E[i]
    acorr = R
    refcoeff = rcoeff
    lpparams = np.concatenate((np.array([1]), -a))
    return acorr, refcoeff, lpparams


def snr(clean_speech, processed_speech, sample_rate):
    # Check the length of the clean and processed speech. Must be the same.
    clean_length = len(clean_speech)
    processed_length = len(processed_speech)
    if clean_length != processed_length:
        raise ValueError("Both Speech Files must be same length.")

    overall_snr = 10 * np.log10(
        np.sum(np.square(clean_speech))
        / np.sum(np.square(clean_speech - processed_speech))
    )

    # Global Variables
    winlength = round(30 * sample_rate / 1000)  # window length in samples
    skiprate = math.floor(winlength / 4)  # window skip in samples
    MIN_SNR = -10  # minimum SNR in dB
    MAX_SNR = 35  # maximum SNR in dB

    # For each frame of input speech, calculate the Segmental SNR
    num_frames = int(
        clean_length / skiprate - (winlength / skiprate)
    )  # number of frames
    start = 0  # starting sample
    window = 0.5 * (
        1 - np.cos(2 * math.pi * np.arange(1, winlength + 1) / (winlength + 1))
    )

    segmental_snr = np.empty(num_frames)
    EPS = np.spacing(1)
    for frame_count in range(num_frames):
        # (1) Get the Frames for the test and reference speech. Multiply by Hanning Window.
        clean_frame = clean_speech[start : start + winlength]
        processed_frame = processed_speech[start : start + winlength]
        clean_frame = np.multiply(clean_frame, window)
        processed_frame = np.multiply(processed_frame, window)

        # (2) Compute the Segmental SNR
        signal_energy = np.sum(np.square(clean_frame))
        noise_energy = np.sum(np.square(clean_frame - processed_frame))
        segmental_snr[frame_count] = 10 * math.log10(
            signal_energy / (noise_energy + EPS) + EPS
        )
        segmental_snr[frame_count] = max(segmental_snr[frame_count], MIN_SNR)
        segmental_snr[frame_count] = min(segmental_snr[frame_count], MAX_SNR)

        start = start + skiprate

    return overall_snr, segmental_snr


def stoi(x, y, fs_signal):
    if np.size(x) != np.size(y):
        raise ValueError("x and y should have the same length")

    # initialization, pay attention to the range of x and y(divide by 32768?)
    fs = 10000  # sample rate of proposed intelligibility measure
    N_frame = 256  # window support
    K = 512  # FFT size
    J = 15  # Number of 1/3 octave bands
    mn = 150  # Center frequency of first 1/3 octave band in Hz
    H, _ = thirdoct(fs, K, J, mn)  # Get 1/3 octave band matrix
    N = 30  # Number of frames for intermediate intelligibility measure (Length analysis window)
    Beta = -15  # lower SDR-bound
    dyn_range = 40  # speech dynamic range

    # resample signals if other sample rate is used than fs
    if fs_signal != fs:
        x = signal.resample_poly(x, fs, fs_signal)
        y = signal.resample_poly(y, fs, fs_signal)

    # remove silent frames
    x, y = removeSilentFrames(x, y, dyn_range, N_frame, int(N_frame / 2))

    # apply 1/3 octave band TF-decomposition
    x_hat = stdft(x, N_frame, N_frame / 2, K)  # apply short-time DFT to clean speech
    y_hat = stdft(
        y, N_frame, N_frame / 2, K
    )  # apply short-time DFT to processed speech

    x_hat = np.transpose(
        x_hat[:, 0 : (int(K / 2) + 1)]
    )  # take clean single-sided spectrum
    y_hat = np.transpose(
        y_hat[:, 0 : (int(K / 2) + 1)]
    )  # take processed single-sided spectrum

    X = np.sqrt(
        np.matmul(H, np.square(np.abs(x_hat)))
    )  # apply 1/3 octave bands as described in Eq.(1) [1]
    Y = np.sqrt(np.matmul(H, np.square(np.abs(y_hat))))

    # loop al segments of length N and obtain intermediate intelligibility measure for all TF-regions
    d_interm = np.zeros(np.size(np.arange(N - 1, x_hat.shape[1])))
    # init memory for intermediate intelligibility measure
    c = 10 ** (-Beta / 20)
    # constant for clipping procedure

    for m in range(N - 1, x_hat.shape[1]):
        X_seg = X[
            :, (m - N + 1) : (m + 1)
        ]  # region with length N of clean TF-units for all j
        Y_seg = Y[
            :, (m - N + 1) : (m + 1)
        ]  # region with length N of processed TF-units for all j
        # obtain scale factor for normalizing processed TF-region for all j
        alpha = np.sqrt(
            np.divide(
                np.sum(np.square(X_seg), axis=1, keepdims=True),
                np.sum(np.square(Y_seg), axis=1, keepdims=True),
            )
        )
        # obtain \alpha*Y_j(n) from Eq.(2) [1]
        aY_seg = np.multiply(Y_seg, alpha)
        # apply clipping from Eq.(3)
        Y_prime = np.minimum(aY_seg, X_seg + X_seg * c)
        # obtain correlation coeffecient from Eq.(4) [1]
        d_interm[m - N + 1] = taa_corr(X_seg, Y_prime) / J

    d = (
        d_interm.mean()
    )  # combine all intermediate intelligibility measures as in Eq.(4) [1]
    return d


def thirdoct(fs, N_fft, numBands, mn):
    """
    [A CF] = THIRDOCT(FS, N_FFT, NUMBANDS, MN) returns 1/3 octave band matrix
    inputs:
        FS:         samplerate
        N_FFT:      FFT size
        NUMBANDS:   number of bands
        MN:         center frequency of first 1/3 octave band
    outputs:
        A:          octave band matrix
        CF:         center frequencies
    """
    f = np.linspace(0, fs, N_fft + 1)
    f = f[0 : int(N_fft / 2 + 1)]
    k = np.arange(numBands)
    cf = np.multiply(np.power(2, k / 3), mn)
    fl = np.sqrt(
        np.multiply(
            np.multiply(np.power(2, k / 3), mn),
            np.multiply(np.power(2, (k - 1) / 3), mn),
        )
    )
    fr = np.sqrt(
        np.multiply(
            np.multiply(np.power(2, k / 3), mn),
            np.multiply(np.power(2, (k + 1) / 3), mn),
        )
    )
    A = np.zeros((numBands, len(f)))

    for i in range(np.size(cf)):
        b = np.argmin((f - fl[i]) ** 2)
        fl[i] = f[b]
        fl_ii = b

        b = np.argmin((f - fr[i]) ** 2)
        fr[i] = f[b]
        fr_ii = b
        A[i, fl_ii:fr_ii] = 1

    rnk = np.sum(A, axis=1)
    end = np.size(rnk)
    rnk_back = rnk[1:end]
    rnk_before = rnk[0 : (end - 1)]
    for i in range(np.size(rnk_back)):
        if (rnk_back[i] >= rnk_before[i]) and (rnk_back[i] != 0):
            result = i
    numBands = result + 2
    A = A[0:numBands, :]
    cf = cf[0:numBands]
    return A, cf


def stdft(x, N, K, N_fft):
    """
    X_STDFT = X_STDFT(X, N, K, N_FFT) returns the short-time hanning-windowed dft of X with frame-size N,
    overlap K and DFT size N_FFT. The columns and rows of X_STDFT denote the frame-index and dft-bin index,
    respectively.
    """
    frames_size = int((np.size(x) - N) / K)
    w = signal.windows.hann(N + 2)
    w = w[1 : N + 1]

    x_stdft = signal.stft(
        x,
        window=w,
        nperseg=N,
        noverlap=K,
        nfft=N_fft,
        return_onesided=False,
        boundary=None,
    )[2]
    x_stdft = np.transpose(x_stdft)[0:frames_size, :]

    return x_stdft


def removeSilentFrames(x, y, dyrange, N, K):
    """
    [X_SIL Y_SIL] = REMOVESILENTFRAMES(X, Y, RANGE, N, K) X and Y are segmented with frame-length N
    and overlap K, where the maximum energy of all frames of X is determined, say X_MAX.
    X_SIL and Y_SIL are the reconstructed signals, excluding the frames, where the energy of a frame
    of X is smaller than X_MAX-RANGE
    """

    frames = np.arange(0, (np.size(x) - N), K)
    w = signal.windows.hann(N + 2)
    w = w[1 : N + 1]

    jj_list = np.empty((np.size(frames), N), dtype=int)
    for j in range(np.size(frames)):
        jj_list[j, :] = np.arange(frames[j] - 1, frames[j] + N - 1)

    msk = 20 * np.log10(np.divide(norm(np.multiply(x[jj_list], w), axis=1), np.sqrt(N)))

    msk = (msk - np.max(msk) + dyrange) > 0
    count = 0

    x_sil = np.zeros(np.size(x))
    y_sil = np.zeros(np.size(y))

    for j in range(np.size(frames)):
        if msk[j]:
            jj_i = np.arange(frames[j], frames[j] + N)
            jj_o = np.arange(frames[count], frames[count] + N)
            x_sil[jj_o] = x_sil[jj_o] + np.multiply(x[jj_i], w)
            y_sil[jj_o] = y_sil[jj_o] + np.multiply(y[jj_i], w)
            count = count + 1

    x_sil = x_sil[0 : jj_o[-1] + 1]
    y_sil = y_sil[0 : jj_o[-1] + 1]
    return x_sil, y_sil


def taa_corr(x, y):
    """
    RHO = TAA_CORR(X, Y) Returns correlation coeffecient between column
    vectors x and y. Gives same results as 'corr' from statistics toolbox.
    """
    xn = np.subtract(x, np.mean(x, axis=1, keepdims=True))
    xn = np.divide(xn, norm(xn, axis=1, keepdims=True))
    yn = np.subtract(y, np.mean(y, axis=1, keepdims=True))
    yn = np.divide(yn, norm(yn, axis=1, keepdims=True))
    rho = np.trace(np.matmul(xn, np.transpose(yn)))

    return rho


In [3]:
import importlib
import time
import os
from pesq import pesq
from pystoi.stoi import stoi
import torch
import numpy as np



def sample_fixed_length_data_aligned_three(data_a, data_b, data_c, sample_length):
    """sample with fixed length from two dataset
    """
    # Replicate data_a and data_b until their lengths are >= sample_length
    while len(data_a) < sample_length:
        data_a = np.concatenate((data_a, data_a), axis=0)
        data_b = np.concatenate((data_b, data_b), axis=0)
        data_c = np.concatenate((data_c, data_c), axis=0)

    frames_total = len(data_a)

    start = np.random.randint(frames_total - sample_length + 1)
    # print(f"Random crop from: {start}")
    end = start + sample_length

    return data_a[start:end], data_b[start:end], data_c[start:end]
    
def sample_fixed_length_data_alignedval13(data_a, data_b, data_c, sample_length):
    """sample with fixed length from two dataset
    """
    assert len(data_a) == len(data_b), "Inconsistent dataset length, unable to sampling"
    # Replicate data_a and data_b until their lengths are >= sample_length
    while len(data_a) < 2*sample_length:
        data_a = np.concatenate((data_a, data_a), axis=0)
        data_b = np.concatenate((data_b, data_b), axis=0)
        data_c = np.concatenate((data_c, data_c), axis=0)

    # Truncate data_a and data_b to ensure their lengths match sample_length
    data_a1 = data_a[:sample_length]
    data_b1 = data_b[:sample_length]
    data_c1 = data_c[:sample_length]

    data_a2 = data_a[sample_length: 2*sample_length]
    data_b2 = data_b[sample_length: 2*sample_length]
    data_c2 = data_c[sample_length: 2*sample_length]

    frames_total = len(data_a)


    return data_a1, data_b1, data_c1, data_a2, data_b2, data_c2


def load_checkpoint(checkpoint_path, device):
    _, ext = os.path.splitext(os.path.basename(checkpoint_path))
    assert ext in (".pth", ".tar"), "Only support ext and tar extensions of model checkpoint."
    model_checkpoint = torch.load(checkpoint_path, map_location=device)

    if ext == ".pth":
        print(f"Loading {checkpoint_path}.")
        return model_checkpoint
    else:  # tar
        print(f"Loading {checkpoint_path}, epoch = {model_checkpoint['epoch']}.")
        return model_checkpoint["model"]


def prepare_empty_dir(dirs, resume=False):
    """
    if resume experiment, assert the dirs exist,
    if not resume experiment, make dirs.

    Args:
        dirs (list): directors list
        resume (bool): whether to resume experiment, default is False
    """
    for dir_path in dirs:
        if resume:
            assert dir_path.exists()
        else:
            dir_path.mkdir(parents=True, exist_ok=True)


class ExecutionTime:
    """
    Usage:
        timer = ExecutionTime()
        <Something...>
        print(f'Finished in {timer.duration()} seconds.')
    """

    def __init__(self):
        self.start_time = time.time()

    def duration(self):
        return int(time.time() - self.start_time)


def compute_PESQ(clean_signal, noisy_signal, sr=16000):
    return pesq(sr, clean_signal, noisy_signal, "wb")


def z_score(m):
    mean = np.mean(m)
    std_var = np.std(m)
    return (m - mean) / std_var, mean, std_var


def reverse_z_score(m, mean, std_var):
    return m * std_var + mean


def min_max(m):
    m_max = np.max(m)
    m_min = np.min(m)

    return (m - m_min) / (m_max - m_min), m_max, m_min


def reverse_min_max(m, m_max, m_min):
    return m * (m_max - m_min) + m_min



def sample_fixed_length_data_aligned(data_a, data_b, sample_length):
    """sample with fixed length from two dataset
    """
    # Replicate data_a and data_b until their lengths are >= sample_length
    while len(data_a) < sample_length:
        data_a = np.concatenate((data_a, data_a), axis=0)
        data_b = np.concatenate((data_b, data_b), axis=0)

    frames_total = len(data_a)

    start = np.random.randint(frames_total - sample_length + 1)
    # print(f"Random crop from: {start}")
    end = start + sample_length

    return data_a[start:end], data_b[start:end]

def sample_fixed_length_data_alignedval1(data_a, data_b, sample_length):
    """sample with fixed length from two dataset
    """
    assert len(data_a) == len(data_b), "Inconsistent dataset length, unable to sampling"
    # Replicate data_a and data_b until their lengths are >= sample_length
    while len(data_a) < 2*sample_length:
        data_a = np.concatenate((data_a, data_a), axis=0)
        data_b = np.concatenate((data_b, data_b), axis=0)

    # Truncate data_a and data_b to ensure their lengths match sample_length
    data_a1 = data_a[:sample_length]
    data_b1 = data_b[:sample_length]

    data_a2 = data_a[sample_length: 2*sample_length]
    data_b2 = data_b[sample_length: 2*sample_length]

    frames_total = len(data_a)


    return data_a1, data_b1, data_a2, data_b2


def sample_fixed_length_data_alignedval2(data_a_list, data_b_list, sample_length):
    """sample with fixed length from two dataset
    """
    ans1 = []
    ans2 = []
    for data_a in data_a_list:
        while len(data_a) < 2*sample_length:
            data_a = np.concatenate((data_a, data_a), axis=0)

        # Truncate data_a and data_b to ensure their lengths match sample_length
        data_a1 = data_a[:sample_length]

        data_a2 = data_a[sample_length: 2*sample_length]

        ans1.append(data_a1.reshape(1, -1))
        ans2.append(data_a2.reshape(1, -1))

    return ans1, ans2


def compute_STOI(clean_signal, noisy_signal, sr=16000):
    return stoi(clean_signal, noisy_signal, sr, extended=False)


def print_tensor_info(tensor, flag="Tensor"):
    floor_tensor = lambda float_tensor: int(float(float_tensor) * 1000) / 1000
    print(flag)
    print(
        f"\tmax: {floor_tensor(torch.max(tensor))}, min: {float(torch.min(tensor))}, mean: {floor_tensor(torch.mean(tensor))}, std: {floor_tensor(torch.std(tensor))}")




from torch.utils.tensorboard import SummaryWriter


def writer(logs_dir):
    return SummaryWriter(log_dir=logs_dir, max_queue=5, flush_secs=30)





2025-12-19 20:40:08.571797: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-19 20:40:10.514364: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1766176811.000763    1361 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1766176811.234644    1361 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-12-19 20:40:12.810194: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

In [4]:
import os

import librosa
from torch.utils import data
import random
import torchaudio.transforms as T
import torch.nn.functional as F

class Dataset(data.Dataset):
    def __init__(self,
                 dataset,
                 limit,
                 offset=0,
                 sample_length=32000,
                 mode="train"):
        """Construct dataset for training and validation.
        Args:
            dataset (str): *.txt, the path of the dataset list file. See "Notes."
            limit (int): Return at most limit files in the list. If None, all files are returned.
            offset (int): Return files starting at an offset within the list. Use negative values to offset from the end of the list.
            sample_length(int): The model only supports fixed-length input. Use sample_length to specify the feature size of the input.
            mode(str): If mode is "train", return fixed-length signals. If mode is "validation", return original-length signals.

        Notes:
            dataset list file：
            <noisy_1_path><space><clean_1_path>
            <noisy_2_path><space><clean_2_path>
            ...
            <noisy_n_path><space><clean_n_path>

            e.g.
            /train/noisy/a.wav /train/clean/a.wav
            /train/noisy/b.wav /train/clean/b.wav
            ...

        Return:
            (mixture signals, clean signals, filename)
        """
        super(Dataset, self).__init__()
        dataset_list = [line.rstrip('\n') for line in open(os.path.abspath(os.path.expanduser(dataset)), "r")]

        dataset_list = dataset_list[offset:limit]

        assert mode in ("train", "validation"), "Mode must be one of 'train' or 'validation'."
        enhanced_dir="./enhanced"
        self.length = len(dataset_list)
        self.dataset_list = dataset_list
        self.sample_length = sample_length
        self.mode = mode

        new_dataset_list = []
        for line in self.dataset_list:

            noisy_path, clean_path = line.split(" ")
            filename = os.path.splitext(os.path.basename(noisy_path))[0]
            enhanced_path = os.path.join(enhanced_dir, f"{filename}_enhanced.wav")
            new_dataset_list.append(f"{noisy_path} {clean_path} {enhanced_path}")

        self.dataset_list = new_dataset_list

    def __len__(self):
        return self.length


    def __getitem__(self, item):
        mixture_path, clean_path, enhance_path = self.dataset_list[item].split(" ")
        filename = os.path.splitext(os.path.basename(mixture_path))[0]
        mixture, _ = librosa.load(os.path.abspath(os.path.expanduser(mixture_path)), sr=16000)
        clean, _ = librosa.load(os.path.abspath(os.path.expanduser(clean_path)), sr=16000)
        enhanced, _ = librosa.load(os.path.abspath(os.path.expanduser(enhance_path)), sr=16000)
        person_id = clean_path.split('/')[-1].split('_')[0]
        same_person_clean_paths = [line.split()[1] for line in self.dataset_list if person_id in line]
        same_person_clean_paths.remove(clean_path)

        same_person_clean_paths = same_person_clean_paths[:50]
        clean_ref_segments = []
        for ref_path in same_person_clean_paths:
            ref_waveform, _ = librosa.load(os.path.abspath(os.path.expanduser(ref_path)), sr=16000)
            clean_ref_segments.append(ref_waveform)

        clean_ref = np.concatenate(clean_ref_segments)
        # aligned_clean_ref = align_with_reference_triplet_top2(clean, clean_ref)


        # aligned_clean_ref = align_with_speech(clean, clean_ref_segments)

        # clean_ref_path = random.choice(same_person_clean_paths)
        # aligned_clean_ref, _ = librosa.load(os.path.abspath(os.path.expanduser(clean_ref_path)), sr=16000)


        if self.mode == "train":
            # The input of model should be fixed-length in the training.
            mixture, clean, enhanc = sample_fixed_length_data_aligned_three(mixture, clean, enhanced, self.sample_length)
            aligned_clean_ref = align_with_reference_quintet_matmul_weighted(enhanc, clean_ref)
            mixture2, clean_ref1 = sample_fixed_length_data_aligned(aligned_clean_ref, aligned_clean_ref, self.sample_length)
            return mixture.reshape(1, -1), clean.reshape(1, -1), clean_ref1.reshape(1, -1), filename


        else:


            mixture1, clean1, enhanc1, mixture2, clean2, enhanc2 = sample_fixed_length_data_alignedval13(mixture, clean, enhanced, self.sample_length)
            
            aligned_vector1 = align_with_reference_quintet_matmul_weighted(enhanc1, clean_ref)
            aligned_vector2 = align_with_reference_quintet_matmul_weighted(enhanc2, clean_ref)
            all_ref1, all_ref2, aa, bb = sample_fixed_length_data_alignedval1(aligned_vector1, aligned_vector2, self.sample_length)

            useABSclean = False
            if useABSclean:
                return mixture1.reshape(1, -1), clean1.reshape(1, -1), clean1.reshape(1, -1), mixture2.reshape(1, -1),clean2.reshape(1, -1), clean2.reshape(1, -1), filename
            else:
                return mixture1.reshape(1, -1), clean1.reshape(1, -1), all_ref1.reshape(1, -1), mixture2.reshape(1, -1),clean2.reshape(1, -1), all_ref2.reshape(1, -1), filename





import numpy as np
from numpy.lib.stride_tricks import sliding_window_view


def align_with_reference_quintet_matmul_weighted(
    clean,
    clean_ref,
    sample_rate=16000,
    n_fft=512,
    hop_length=256,
    win_length=512,
):
    """
    Align `clean` to `clean_ref` by matching weighted quintets of STFT magnitudes
    (weights = [0.25, 0.5, 1.0, 0.5, 0.25] on [i-2, i-1, i, i+1, i+2] frames),
    using matrix multiplication for all interior‐frame similarities.  The first,
    second, second‐last, and last frames are matched by single‐frame best cosine
    similarity.
    """
    # STFT / ISTFT transforms
    transform = T.Spectrogram(
        n_fft=n_fft, hop_length=hop_length, win_length=win_length, power=None
    )
    istft = T.InverseSpectrogram(
        n_fft=n_fft, hop_length=hop_length, win_length=win_length
    )

    # to float tensors
    clean_t = torch.tensor(clean,     dtype=torch.float32)
    ref_t   = torch.tensor(clean_ref, dtype=torch.float32)

    # compute complex spectrograms: [freq, time]
    spec1 = transform(clean_t)
    spec2 = transform(ref_t)

    # magnitudes and L2‐normalize across freq‐axis
    mag1 = spec1.abs()
    mag2 = spec2.abs()
    mag1n = F.normalize(mag1, p=2, dim=0)  # [freq, T1]
    mag2n = F.normalize(mag2, p=2, dim=0)  # [freq, T2]

    T1 = mag1n.size(1)
    T2 = mag2n.size(1)
    assert T1 >= 5 and T2 >= 5, "Need at least 5 frames in each signal."

    matched_specs = []

    # 1) Match first frame (i=0) and second frame (i=1) by single‐frame similarity
    for i in (0, 1):
        wi   = mag1n[:, i]               # [freq]
        sims = mag2n.T.matmul(wi)        # [T2]
        best = sims.argmax().item()
        matched_specs.append(spec2[:, best].unsqueeze(1))

    # 2) Interior quintets for i = 2 .. T1-3
    #    unfold into sliding windows of width=5 → [freq, T1-4, 5]
    quint1 = mag1n.unfold(dimension=1, size=5, step=1)  # [freq, T1-4, 5]
    quint2 = mag2n.unfold(dimension=1, size=5, step=1)  # [freq, T2-4, 5]

    # apply weights [0.25, 0.5, 1.0, 0.5, 0.25]
    weights = torch.tensor(
        [0.15, 0.4, 1.0, 0.4, 0.15],
        dtype=mag1n.dtype,
        device=mag1n.device
    ).view(1, 1, 5)
    quint1 = quint1 * weights
    quint2 = quint2 

    # flatten for bulk matmul
    # quint1_flat: [T1-4, freq*5]
    quint1_flat = quint1.permute(1, 0, 2).reshape(T1 - 4, -1)
    # quint2_flat: [freq*5, T2-4]
    quint2_flat = quint2.permute(0, 2, 1).reshape(-1, T2 - 4)

    # compute all pairwise similarities: [T1-4, T2-4]
    sims_mat = quint1_flat.matmul(quint2_flat)

    # for each query quintet (row) pick best column → middle index = col + 2
    best_js = sims_mat.argmax(dim=1) + 2  # shape: [T1-4]

    # gather the matched middle‐frame complex spectra
    for j in best_js.tolist():
        matched_specs.append(spec2[:, j].unsqueeze(1))

    # 3) Match second‐last (i=T1-2) and last (i=T1-1) by single‐frame similarity
    for i in (T1 - 2, T1 - 1):
        wi   = mag1n[:, i]
        sims = mag2n.T.matmul(wi)
        best = sims.argmax().item()
        matched_specs.append(spec2[:, best].unsqueeze(1))

    # 4) Concatenate matched frames and invert STFT
    aligned_spec = torch.cat(matched_specs, dim=1)  # [freq, T1]
    aligned_wave = istft(aligned_spec)              # [time]

    return aligned_wave.cpu().numpy()



def align_with_reference_triplet_matmul_weighted(
    clean,
    clean_ref,
    sample_rate=16000,
    n_fft=512,
    hop_length=256,
    win_length=512,
):
    """
    Align `clean` to `clean_ref` by matching weighted triplets of STFT magnitudes
    (weights = [0.5, 1.0, 0.5] on [prev, mid, next] frames), using a single matmul
    for all interior-frame similarities.
    """
    # STFT / ISTFT
    transform = T.Spectrogram(n_fft=n_fft, hop_length=hop_length,
                              win_length=win_length, power=None)
    istft = T.InverseSpectrogram(n_fft=n_fft, hop_length=hop_length,
                                 win_length=win_length)

    # to tensors
    clean_t = torch.tensor(clean, dtype=torch.float32)
    ref_t   = torch.tensor(clean_ref, dtype=torch.float32)

    # compute complex spectrograms [freq, time]
    spec1 = transform(clean_t)
    spec2 = transform(ref_t)

    # magnitudes + L2 normalize across freq
    mag1 = spec1.abs()
    mag2 = spec2.abs()
    mag1n = F.normalize(mag1, p=2, dim=0)
    mag2n = F.normalize(mag2, p=2, dim=0)

    matched_specs = []

    # 1) first frame: simple matmul
    w0     = mag1n[:, 0]                 # [freq]
    sims0   = mag2n.T.matmul(w0)         # [time2]
    best0   = sims0.argmax().item()
    matched_specs.append(spec2[:, best0].unsqueeze(1))

    # 2) interior triplets
    T1 = mag1n.shape[1]
    T2 = mag2n.shape[1]
    assert T1 >= 3 and T2 >= 3, "Need ≥3 frames"

    # unfold into sliding windows of width=3
    # yields shape [freq, T-2, 3]
    trip1 = mag1n.unfold(dimension=1, size=3, step=1)
    trip2 = mag2n.unfold(dimension=1, size=3, step=1)

    # apply weights [0.5, 1.0, 0.5] on the last dim
    weights = torch.tensor([0.5, 1.0, 0.5],
                           device=mag1n.device,
                           dtype=mag1n.dtype).view(1, 1, 3)
    trip1 = trip1 * weights
    trip2 = trip2 

    # flatten into matrices for matmul
    # trip1_flat: [T1-2, freq*3]
    trip1_flat = trip1.permute(1, 0, 2).reshape(T1-2, -1)
    # trip2_flat: [freq*3, T2-2]
    trip2_flat = trip2.permute(0, 2, 1).reshape(-1, T2-2)

    # one big matmul → [T1-2, T2-2]
    sims = trip1_flat.matmul(trip2_flat)

    # for each query row pick best column → middle index = col+1
    best_js = sims.argmax(dim=1) + 1       # shape [T1-2]

    # gather matched spec2 frames
    for j in best_js.tolist():
        matched_specs.append(spec2[:, j].unsqueeze(1))

    # 3) last frame
    wL     = mag1n[:, -1]
    simsL   = mag2n.T.matmul(wL)
    bestL   = simsL.argmax().item()
    matched_specs.append(spec2[:, bestL].unsqueeze(1))

    # 4) concat & ISTFT
    aligned_spec = torch.cat(matched_specs, dim=1)
    aligned_wave = istft(aligned_spec)

    return aligned_wave.cpu().numpy()

def align_with_reference_old(clean, clean_ref, sample_rate=16000, n_fft=512, hop_length=256, win_length=512):
    transform = T.Spectrogram(n_fft=n_fft, hop_length=hop_length, win_length=win_length, power=None)
    istft = T.InverseSpectrogram(n_fft=n_fft, hop_length=hop_length, win_length=win_length)


    # Convert to tensors
    clean_tensor = torch.tensor(clean)
    ref_tensor = torch.tensor(clean_ref)
    # STFTs
    spec1 = transform(clean_tensor)
    spec2 = transform(ref_tensor)

    # Magnitude
    mag1 = spec1.abs()
    mag2 = spec2.abs()

    # Normalize
    mag1_norm = F.normalize(mag1, p=2, dim=0)
    mag2_norm = F.normalize(mag2, p=2, dim=0)

    # Match windows
    matched = []
    for i in range(mag1_norm.shape[1]):
        w1 = mag1_norm[:, i].unsqueeze(1)
        sim = torch.matmul(mag2_norm.transpose(0, 1), w1).squeeze()
        best_idx = torch.argmax(sim).item()
        matched.append(spec2[:, best_idx].unsqueeze(1))

    # Concatenate matched windows and reconstruct
    aligned_spec = torch.cat(matched, dim=1)
    aligned_waveform = istft(aligned_spec)
    return aligned_waveform.numpy()





In [5]:
import torch
import torch.nn as nn
import numpy as np
import torch.nn.functional as F
from scipy.signal import get_window


def init_kernels(win_len, win_inc, fft_len, win_type=None, invers=False):
    if win_type == 'None' or win_type is None:
        window = np.ones(win_len)
    else:
        window = get_window(win_type, win_len, fftbins=True)

    N = fft_len
    fourier_basis = np.fft.rfft(np.eye(N))[:win_len]
    real_kernel = np.real(fourier_basis)
    imag_kernel = np.imag(fourier_basis)
    kernel = np.concatenate([real_kernel, imag_kernel], 1).T

    if invers :
        kernel = np.linalg.pinv(kernel).T

    kernel = kernel*window
    kernel = kernel[:, None, :]
    return torch.from_numpy(kernel.astype(np.float32)), torch.from_numpy(window[None,:,None].astype(np.float32))


class ConvSTFT(nn.Module):

    def __init__(self, win_len, win_inc, fft_len=None, win_type='hamming', feature_type='real', fix=True):
        super(ConvSTFT, self).__init__()

        if fft_len == None:
            self.fft_len = np.int(2**np.ceil(np.log2(win_len)))
        else:
            self.fft_len = fft_len

        kernel, _ = init_kernels(win_len, win_inc, self.fft_len, win_type)
        #self.weight = nn.Parameter(kernel, requires_grad=(not fix))
        self.register_buffer('weight', kernel)
        self.feature_type = feature_type
        self.stride = win_inc
        self.win_len = win_len
        self.dim = self.fft_len

    def forward(self, inputs):
        if inputs.dim() == 2:
            inputs = torch.unsqueeze(inputs, 1)
        inputs = F.pad(inputs,[self.win_len-self.stride, self.win_len-self.stride])
        outputs = F.conv1d(inputs, self.weight, stride=self.stride)
        dim = self.dim//2+1
        real = outputs[:, :dim, :]
        imag = outputs[:, dim:, :]
        mags = torch.sqrt(real**2+imag**2)
        phase = torch.atan2(imag, real)
        return mags, phase, outputs

class ConviSTFT(nn.Module):

    def __init__(self, win_len, win_inc, fft_len=None, win_type='hamming', feature_type='real', fix=True):
        super(ConviSTFT, self).__init__()
        if fft_len == None:
            self.fft_len = np.int(2**np.ceil(np.log2(win_len)))
        else:
            self.fft_len = fft_len
        kernel, window = init_kernels(win_len, win_inc, self.fft_len, win_type, invers=True)
        #self.weight = nn.Parameter(kernel, requires_grad=(not fix))
        self.register_buffer('weight', kernel)
        self.feature_type = feature_type
        self.win_type = win_type
        self.win_len = win_len
        self.stride = win_inc
        self.stride = win_inc
        self.dim = self.fft_len
        self.register_buffer('window', window)
        self.register_buffer('enframe', torch.eye(win_len)[:,None,:])

    def forward(self, inputs, phase = None):
        """
        inputs : [B, N+2, T] (complex spec)
        or [B, N//2+1, T] (mags)
        phase: [B, N//2+1, T] (if not none)
        """

        if phase is not None:
            real = inputs*torch.cos(phase)
            imag = inputs*torch.sin(phase)
            inputs = torch.cat([real, imag], 1)
        outputs = F.conv_transpose1d(inputs, self.weight, stride=self.stride)

        # this is from torch-stft: https://github.com/pseeth/torch-stft
        t = self.window.repeat(1,1,inputs.size(-1))**2
        coff = F.conv_transpose1d(t, self.enframe, stride=self.stride)
        outputs = outputs/(coff+1e-8)
        #outputs = torch.where(coff == 0, outputs, outputs/coff)
        outputs = outputs[...,self.win_len-self.stride:-(self.win_len-self.stride)]

        return outputs


import librosa
import soundfile as sf








In [6]:
import torch.nn as nn
import torch
import torch.nn.functional as F
import os
import sys

def mse_loss():
    return torch.nn.MSELoss()

def l1_loss():
    return torch.nn.L1Loss()

def CE_loss():
    return torch.nn.CrossEntropyLoss()

class GLayerNorm2d(nn.Module):

    def __init__(self, in_channel, eps=1e-12):
        super(GLayerNorm2d, self).__init__()
        self.eps = eps
        self.beta = nn.Parameter(torch.ones([1, in_channel,1,1]))
        self.gamma = nn.Parameter(torch.zeros([1, in_channel,1,1]))

    def forward(self,inputs):
        mean = torch.mean(inputs,[1,2,3], keepdim=True)
        var = torch.var(inputs,[1,2,3], keepdim=True)
        outputs = (inputs - mean)/ torch.sqrt(var+self.eps)*self.beta+self.gamma
        return outputs

class Axial_Layer(nn.Module):
    def __init__(self, in_channels, num_heads=8, kernel_size=7, stride=1, height_dim=True, inference=False):
        super(Axial_Layer, self).__init__()
        self.depth = in_channels
        self.num_heads = num_heads
        self.kernel_size = kernel_size
        self.stride = stride
        self.height_dim = height_dim
        self.dh = self.depth // self.num_heads

        assert self.depth % self.num_heads == 0, "depth should be divided by num_heads. (example: depth: 32, num_heads: 8)"

        self.kqv_conv = nn.Conv1d(in_channels, self.depth * 2, kernel_size=1, bias=False)
        self.kqv_bn = nn.BatchNorm1d(self.depth * 2)
        self.logits_bn = nn.BatchNorm2d(num_heads * 3)
        # Positional encodings
        self.rel_encoding = nn.Parameter(torch.randn(self.dh * 2, kernel_size * 2 - 1), requires_grad=True)
        key_index = torch.arange(kernel_size)
        query_index = torch.arange(kernel_size)
        # Shift the distance_matrix so that it is >= 0. Each entry of the
        # distance_matrix distance will index a relative positional embedding.
        distance_matrix = (key_index[None, :] - query_index[:, None]) + kernel_size - 1
        self.register_buffer('distance_matrix', distance_matrix.reshape(kernel_size*kernel_size))

        # later access attention weights
        self.inference = inference
        if self.inference:
            self.register_parameter('weights', None)

    def forward(self, x):

        if self.height_dim:
            x = x.permute(0, 3, 1, 2)  # batch_size, width, depth, height
        else:
            x = x.permute(0, 2, 1, 3)  # batch_size, height, depth, width

        batch_size, width, depth, height = x.size()
        x = x.reshape(batch_size * width, depth, height)
        # Compute q, k, v
        kqv = self.kqv_conv(x)
        kqv = self.kqv_bn(kqv) # apply batch normalization on k, q, v
        k, q, v = torch.split(kqv.reshape(batch_size * width, self.num_heads, self.dh * 2, height), [self.dh // 2, self.dh // 2, self.dh], dim=2)
        # Positional encodings
        rel_encodings = torch.index_select(self.rel_encoding, 1, self.distance_matrix).reshape(self.dh * 2, self.kernel_size, self.kernel_size)
        q_encoding, k_encoding, v_encoding = torch.split(rel_encodings, [self.dh // 2, self.dh // 2, self.dh], dim=0)
        # qk + qr + kr
        qk = torch.matmul(q.transpose(2, 3), k)
        qr = torch.einsum('bhdx,dxy->bhxy', q, q_encoding)
        kr = torch.einsum('bhdx,dxy->bhxy', k, k_encoding).transpose(2, 3)
        logits = torch.cat([qk, qr, kr], dim=1)
        logits = self.logits_bn(logits) # apply batch normalization on qk, qr, kr
        logits = logits.reshape(batch_size * width, 3, self.num_heads, height, height).sum(dim=1)
        weights = F.softmax(logits, dim=3)

        if self.inference:
            self.weights = nn.Parameter(weights)

        attn = torch.matmul(weights, v.transpose(2,3)).transpose(2,3)
        attn_encoding = torch.einsum('bhxy,dxy->bhdx', weights, v_encoding)
        attn_out = torch.cat([attn, attn_encoding], dim=-1).reshape(batch_size * width, self.depth * 2, height)
        output = attn_out.reshape(batch_size, width, self.depth, 2, height).sum(dim=-2)

        if self.height_dim:
            output = output.permute(0, 2, 3, 1)
        else:
            output = output.permute(0, 2, 1, 3)

        return output

class Axial_Layer_cross(nn.Module):
    def __init__(self, in_channels, num_heads=8, kernel_size=7, stride=1, height_dim=True, inference=False):
        super(Axial_Layer_cross, self).__init__()
        self.depth = in_channels
        self.num_heads = num_heads
        self.kernel_size = kernel_size
        self.stride = stride
        self.height_dim = height_dim
        self.dh = self.depth // self.num_heads

        assert self.depth % self.num_heads == 0, "depth should be divided by num_heads. (example: depth: 32, num_heads: 8)"

        self.v_conv = nn.Conv1d(in_channels, self.depth, kernel_size=1, bias=False)
        self.v_bn = nn.BatchNorm1d(self.depth)

        self.q_conv = nn.Conv1d(in_channels, self.depth // 2, kernel_size=1, bias=False)
        self.q_bn = nn.BatchNorm1d(self.depth // 2)

        self.k_conv = nn.Conv1d(in_channels, self.depth // 2, kernel_size=1, bias=False)
        self.k_bn = nn.BatchNorm1d(self.depth // 2)


        self.kq_conv = nn.Conv1d(in_channels, self.depth, kernel_size=1, bias=False)
        self.kq_bn = nn.BatchNorm1d(self.depth)

        self.logits_bn = nn.BatchNorm2d(num_heads * 3)
        # Positional encodings
        self.rel_encoding = nn.Parameter(torch.randn(self.dh * 2, kernel_size * 2 - 1), requires_grad=True)
        key_index = torch.arange(kernel_size)
        query_index = torch.arange(kernel_size)
        # Shift the distance_matrix so that it is >= 0. Each entry of the
        # distance_matrix distance will index a relative positional embedding.
        distance_matrix = (key_index[None, :] - query_index[:, None]) + kernel_size - 1
        self.register_buffer('distance_matrix', distance_matrix.reshape(kernel_size*kernel_size))

        # later access attention weights
        self.inference = inference
        if self.inference:
            self.register_parameter('weights', None)

    def forward(self, x, y):
        if self.height_dim:
            x = x.permute(0, 3, 1, 2)  # batch_size, width, depth, height
            y = y.permute(0, 3, 1, 2)  # batch_size, width, depth, height
        else:
            x = x.permute(0, 2, 1, 3)  # batch_size, height, depth, width
            y = y.permute(0, 2, 1, 3)  # batch_size, height, depth, width

        batch_size, width, depth, height = x.size()
        x = x.reshape(batch_size * width, depth, height)
        y = y.reshape(batch_size * width, depth, height)
        # Compute q, k, v
        k = self.k_conv(x)
        k = self.k_bn(k) # apply batch normalization on k, q, v

        v = self.v_conv(x)
        v = self.kq_bn(v) # apply batch normalization on k, q, v

        q = self.q_conv(y)
        q = self.q_bn(q) # apply batch normalization on k, q, v

        kqv = torch.cat([k, q, v], dim = 1)
        k, q, v = torch.split(kqv.reshape(batch_size * width, self.num_heads, self.dh * 2, height), [self.dh // 2, self.dh // 2, self.dh], dim=2)
        #q = q.reshape(batch_size * width, self.num_heads, self.dh, height)

        # Positional encodings
        rel_encodings = torch.index_select(self.rel_encoding, 1, self.distance_matrix).reshape(self.dh * 2, self.kernel_size, self.kernel_size)
        q_encoding, k_encoding, v_encoding = torch.split(rel_encodings, [self.dh // 2, self.dh // 2, self.dh], dim=0)

        # qk + qr + kr
        qk = torch.matmul(q.transpose(2, 3), k)
        qr = torch.einsum('bhdx,dxy->bhxy', q, q_encoding)
        kr = torch.einsum('bhdx,dxy->bhxy', k, k_encoding).transpose(2, 3)

        logits = torch.cat([qk, qr, kr], dim=1)
        logits = self.logits_bn(logits) # apply batch normalization on qk, qr, kr
        logits = logits.reshape(batch_size * width, 3, self.num_heads, height, height).sum(dim=1)

        weights = F.softmax(logits, dim=3)

        if self.inference:
            self.weights = nn.Parameter(weights)

        attn = torch.matmul(weights, v.transpose(2,3)).transpose(2,3)
        attn_encoding = torch.einsum('bhxy,dxy->bhdx', weights, v_encoding)
        attn_out = torch.cat([attn, attn_encoding], dim=-1).reshape(batch_size * width, self.depth * 2, height)
        output = attn_out.reshape(batch_size, width, self.depth, 2, height).sum(dim=-2)

        if self.height_dim:
            output = output.permute(0, 2, 3, 1)
        else:
            output = output.permute(0, 2, 1, 3)

        return output

class CausalConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Conv2d(
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=(3, 2),
            stride=(2, 1),
            padding=(0, 1)
        )
        self.norm = nn.BatchNorm2d(num_features=out_channels)
        self.activation = nn.ELU()

    def forward(self, x):
        """
        2D Causal convolution.
        Args:
            x: [B, C, F, T]

        Returns:
            [B, C, F, T]
        """
        x = self.conv(x)
        x = x[:, :, :, :-1]  # chomp size
        x = self.norm(x)
        x = self.activation(x)
        return x


class CausalTransConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, is_last=False, output_padding=(0, 0)):
        super().__init__()
        self.conv = nn.ConvTranspose2d(
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=(3, 2),
            stride=(2, 1),
            output_padding=output_padding
        )
        self.norm = nn.BatchNorm2d(num_features=out_channels)
        if is_last:
            self.activation = nn.ReLU()
        else:
            self.activation = nn.ELU()

    def forward(self, x):
        """
        2D Causal convolution.
        Args:
            x: [B, C, F, T]

        Returns:
            [B, C, F, T]
        """
        x = self.conv(x)
        x = x[:, :, :, :-1]  # chomp size
        x = self.norm(x)
        x = self.activation(x)
        return x

class Model(nn.Module):

    def __init__(self, channel_amp = 1, channel_phase=3):
        super(Model, self).__init__()
        self.stft = ConvSTFT(512, 256, 512, 'hamming', 'real', True)
        self.istft = ConviSTFT(512, 256, 512, 'hamming', 'real', True)

        # Encoder
        self.conv_block_1 = CausalConvBlock(1, 16)
        self.conv_block_2 = CausalConvBlock(16, 32)
        self.conv_block_3 = CausalConvBlock(32, 64)
        self.conv_block_4 = CausalConvBlock(64, 128)
        self.conv_block_5 = CausalConvBlock(128, 256)

        self.conv_block_1ref = CausalConvBlock(1, 16)
        self.conv_block_2ref = CausalConvBlock(16, 32)
        self.conv_block_3ref = CausalConvBlock(32, 64)
        self.conv_block_4ref = CausalConvBlock(64, 128)
        self.conv_block_5ref = CausalConvBlock(128, 256)

        self.SA_time = Axial_Layer(256, height_dim=True)
        self.SA_frequency = Axial_Layer(256, kernel_size=126, height_dim = False)


        self.CA_time_1 = Axial_Layer_cross(256, height_dim=True)
        self.CA_time_2 = Axial_Layer_cross(128, kernel_size=15, height_dim=True)
        self.CA_time_3 = Axial_Layer_cross(64, kernel_size=31, height_dim=True)
        self.CA_time_4 = Axial_Layer_cross(32, kernel_size=63, height_dim=True)
        self.CA_time_5 = Axial_Layer_cross(16, kernel_size=128, height_dim=True)

        self.CA_time_11 = Axial_Layer_cross(256, height_dim=True)
        self.CA_time_22 = Axial_Layer_cross(128, kernel_size=15, height_dim=True)
        self.CA_time_33 = Axial_Layer_cross(64, kernel_size=31, height_dim=True)
        self.CA_time_44 = Axial_Layer_cross(32, kernel_size=63, height_dim=True)
        self.CA_time_55 = Axial_Layer_cross(16, kernel_size=128, height_dim=True)

        self.CA_frequency_1 = Axial_Layer_cross(256, kernel_size=126, height_dim = False)
        self.CA_frequency_2 = Axial_Layer_cross(128, kernel_size=126, height_dim = False)
        self.CA_frequency_3 = Axial_Layer_cross(64, kernel_size=126, height_dim = False)
        self.CA_frequency_4 = Axial_Layer_cross(32, kernel_size=126, height_dim = False)
        self.CA_frequency_5 = Axial_Layer_cross(16, kernel_size=126, height_dim = False)

        self.CA_frequency_11 = Axial_Layer_cross(256, kernel_size=126, height_dim = False)
        self.CA_frequency_22 = Axial_Layer_cross(128, kernel_size=126, height_dim = False)
        self.CA_frequency_33 = Axial_Layer_cross(64, kernel_size=126, height_dim = False)
        self.CA_frequency_44 = Axial_Layer_cross(32, kernel_size=126, height_dim = False)
        self.CA_frequency_55 = Axial_Layer_cross(16, kernel_size=126, height_dim = False)

        self.tran_conv_block_1 = CausalTransConvBlock(256 + 256, 128)
        self.tran_conv_block_2 = CausalTransConvBlock(128 + 128, 64)
        self.tran_conv_block_3 = CausalTransConvBlock(64 + 64, 32)
        self.tran_conv_block_4 = CausalTransConvBlock(32 + 32, 16, output_padding=(1, 0))
        self.tran_conv_block_5 = CausalTransConvBlock(16 + 16, 1, is_last=True)


        self.tran_conv_block_11 = CausalTransConvBlock(256 + 256, 128)
        self.tran_conv_block_22 = CausalTransConvBlock(128 + 128, 64)
        self.tran_conv_block_33 = CausalTransConvBlock(64 + 64, 32)
        self.tran_conv_block_44 = CausalTransConvBlock(32 + 32, 16, output_padding=(1, 0))
        self.tran_conv_block_55 = CausalTransConvBlock(16 + 16, 1, is_last=True)

        self.tran_conv_block_111 = CausalTransConvBlock(256 + 256, 128)
        self.tran_conv_block_222 = CausalTransConvBlock(128 + 128, 64)
        self.tran_conv_block_333 = CausalTransConvBlock(64 + 64, 32)
        self.tran_conv_block_444 = CausalTransConvBlock(32 + 32, 16, output_padding=(1, 0))
        self.tran_conv_block_555 = CausalTransConvBlock(16 + 16, 1, is_last=True)

        self.tran_conv_block_1111 = CausalTransConvBlock(256 + 256, 128)
        self.tran_conv_block_2222 = CausalTransConvBlock(128 + 128, 64)
        self.tran_conv_block_3333 = CausalTransConvBlock(64 + 64, 32)
        self.tran_conv_block_4444 = CausalTransConvBlock(32 + 32, 16, output_padding=(1, 0))
        self.tran_conv_block_5555 = CausalTransConvBlock(16 + 16, 1, is_last=True)


    def forward(self, x, ref):
        amp_spec, phase_spec, comp_spec = self.stft(x)
        ref_amp_spec, ref_phase_spec, ref_comp_spec = self.stft(ref)

        phase_spec = torch.unsqueeze(phase_spec, 1)
        amp_spec = torch.unsqueeze(amp_spec, 1)
        ref_phase_spec = torch.unsqueeze(ref_phase_spec, 1)
        ref_amp_spec = torch.unsqueeze(ref_amp_spec, 1)
        # amp_spec,phase_spec  = torch.Size([1, 1, 257, 251])


        e1 = self.conv_block_1(amp_spec)
        e2 = self.conv_block_2(e1)
        e3 = self.conv_block_3(e2)
        e4 = self.conv_block_4(e3)
        e5 = self.conv_block_5(e4)



        refe1 = self.conv_block_1ref(ref_amp_spec)
        refe2 = self.conv_block_2ref(refe1)
        refe3 = self.conv_block_3ref(refe2)
        refe4 = self.conv_block_4ref(refe3)
        refe5 = self.conv_block_5ref(refe4)


        b_time = self.SA_time(e5)
        b_frequency = self.SA_frequency(e5)
        b = e5 + b_time + b_frequency


        e5_1 = self.CA_time_1(e5, b)
        refe5_1 = self.CA_time_11(refe5, b)

        e5_2 = self.CA_frequency_1(e5, b)
        refe5_2= self.CA_frequency_11(refe5, b)

        d1 = self.tran_conv_block_1(torch.cat([b, e5_1], 1))
        d2 = self.tran_conv_block_11(torch.cat([b, refe5_1], 1))
        d3 = self.tran_conv_block_111(torch.cat([b, e5_2], 1))
        d4 = self.tran_conv_block_1111(torch.cat([b, refe5_2], 1))
        d = (d1 + d2 + d3 + d4) / 4

        e4_1 = self.CA_time_2(e4, d)
        refe4_1 = self.CA_time_22(refe4, d)

        e4_2 = self.CA_frequency_2(e4, d)
        refe4_2= self.CA_frequency_22(refe4, d)

        d1 = self.tran_conv_block_2(torch.cat([d, e4_1], 1))
        d2 = self.tran_conv_block_22(torch.cat([d, refe4_1], 1))
        d3 = self.tran_conv_block_222(torch.cat([d, e4_2], 1))
        d4 = self.tran_conv_block_2222(torch.cat([d, refe4_2], 1))
        d = (d1 + d2 + d3 + d4) / 4

        e3_1 = self.CA_time_3(e3, d)
        refe3_1 = self.CA_time_33(refe3, d)

        e3_2 = self.CA_frequency_3(e3, d)
        refe3_2= self.CA_frequency_33(refe3, d)

        d1 = self.tran_conv_block_3(torch.cat([d, e3_1], 1))
        d2 = self.tran_conv_block_33(torch.cat([d, refe3_1], 1))
        d3 = self.tran_conv_block_333(torch.cat([d, e3_2], 1))
        d4 = self.tran_conv_block_3333(torch.cat([d, refe3_2], 1))
        d = (d1 + d2 + d3 + d4) / 4

        e2_1 = self.CA_time_4(e2, d)
        refe2_1 = self.CA_time_44(refe2, d)

        e2_2 = self.CA_frequency_4(e2, d)
        refe2_2= self.CA_frequency_44(refe2, d)

        d1 = self.tran_conv_block_4(torch.cat([d, e2_1], 1))
        d2 = self.tran_conv_block_44(torch.cat([d, refe2_1], 1))
        d3 = self.tran_conv_block_444(torch.cat([d, e2_2], 1))
        d4 = self.tran_conv_block_4444(torch.cat([d, refe2_2], 1))
        d = (d1 + d2 + d3 + d4) / 4

        e1_1 = self.CA_time_5(e1, d)
        refe1_1 = self.CA_time_55(refe1, d)

        e1_2 = self.CA_frequency_5(e1, d)
        refe1_2= self.CA_frequency_55(refe1, d)

        d1 = self.tran_conv_block_5(torch.cat([d, e1_1], 1))
        d2 = self.tran_conv_block_55(torch.cat([d, refe1_1], 1))
        d3 = self.tran_conv_block_555(torch.cat([d, e1_2], 1))
        d4 = self.tran_conv_block_5555(torch.cat([d, refe1_2], 1))
        d = (d1 + d2 + d3 + d4) / 4

        p5 = phase_spec

        p5 = torch.squeeze(p5, 1)
        d = torch.squeeze(d, 1)

        real = d*torch.cos(p5)
        imag = d*torch.sin(p5)
        est_spec = torch.cat([real, imag], 1)
        est_wav = self.istft(est_spec, None)
        est_wav = torch.squeeze(est_wav, 1)
        return est_spec, est_wav



    def loss_snr(self, est, labels):
        if labels.dim() == 3:
            labels = torch.squeeze(labels,1)
        if est.dim() == 3:
            est = torch.squeeze(est,1)
        return -si_snr(est, labels)



def remove_dc(data):
    mean = torch.mean(data, -1, keepdim=True)
    data = data - mean
    return data
def l2_norm(s1, s2):
    #norm = torch.sqrt(torch.sum(s1*s2, 1, keepdim=True))
    #norm = torch.norm(s1*s2, 1, keepdim=True)

    norm = torch.sum(s1*s2, -1, keepdim=True)
    return norm

def si_snr(s1, s2, eps=1e-8):

    # Ensure inputs are PyTorch tensors
    if not isinstance(s1, torch.Tensor):
        s1 = torch.tensor(s1)
    if not isinstance(s2, torch.Tensor):
        s2 = torch.tensor(s2)

    if s1.dim() == 3:
        s1 = torch.squeeze(s1,1)
    if s2.dim() == 3:
        s2 = torch.squeeze(s2,1)

    s1_s2_norm = l2_norm(s1, s2)
    s2_s2_norm = l2_norm(s2, s2)
    s_target = s1_s2_norm / (s2_s2_norm + eps) * s2
    e_nosie = s1 - s_target  # Now both are tensors
    target_norm = l2_norm(s_target, s_target)
    noise_norm = l2_norm(e_nosie, e_nosie)
    snr = 10 * torch.log10((target_norm) / (noise_norm + eps) + eps)
    return torch.mean(snr)


def test_selfattention1():
    torch.manual_seed(20)

    inputs = torch.randn([1, 2 * 16000])
    ref = torch.randn([1, 2 * 16000])
    net = Model()

    out1, out2 = net(inputs, ref)
    print(out1.shape)
    print(out2.shape)



In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SpectralConvergenceLoss(nn.Module):
    """Metric 1: Spectral Convergence Loss"""
    def __init__(self):
        super(SpectralConvergenceLoss, self).__init__()

    def forward(self, x_mag, y_mag):
        return torch.norm(y_mag - x_mag, p="fro") / (torch.norm(y_mag, p="fro") + 1e-8)


class LogSTFTMagnitudeLoss(nn.Module):
    """Metric 2: Log STFT Magnitude Loss"""
    def __init__(self):
        super(LogSTFTMagnitudeLoss, self).__init__()

    def forward(self, x_mag, y_mag):
        return F.l1_loss(torch.log(y_mag + 1e-8), torch.log(x_mag + 1e-8))


class STFTLoss(nn.Module):
    """
    Single-Resolution STFT Loss calculating both Convergence and Log-Mag distance.
    We only care about Magnitude here because the model uses Noisy Phase.
    """
    def __init__(self, fft_size, hop_size, win_length):
        super(STFTLoss, self).__init__()
        self.fft_size = fft_size
        self.hop_size = hop_size
        self.win_length = win_length
        self.window = torch.hann_window(win_length)
        self.sc_loss = SpectralConvergenceLoss()
        self.mag_loss = LogSTFTMagnitudeLoss()

    def forward(self, x, y):
        # x, y: [Batch, Time]
        
        # Ensure window is on the correct device
        if self.window.device != x.device:
            self.window = self.window.to(x.device)

        # Compute STFT
        x_stft = torch.stft(x, self.fft_size, self.hop_size, self.win_length, 
                            window=self.window, return_complex=True)
        y_stft = torch.stft(y, self.fft_size, self.hop_size, self.win_length, 
                            window=self.window, return_complex=True)

        # Extract Magnitude (Ignore Phase)
        x_mag = torch.clamp(x_stft.abs(), min=1e-8)
        y_mag = torch.clamp(y_stft.abs(), min=1e-8)

        # Compute Losses
        sc_loss = self.sc_loss(x_mag, y_mag)
        mag_loss = self.mag_loss(x_mag, y_mag)

        return sc_loss, mag_loss


class MultiResolutionSTFTLoss(nn.Module):
    """
    Aggregates STFT losses over multiple resolutions.
    """
    def __init__(self, 
                 fft_sizes=[1024, 2048, 512], 
                 hop_sizes=[120, 240, 50], 
                 win_lengths=[600, 1200, 240]):
        super(MultiResolutionSTFTLoss, self).__init__()
        self.stft_losses = nn.ModuleList()
        
        for fs, hs, wl in zip(fft_sizes, hop_sizes, win_lengths):
            self.stft_losses.append(STFTLoss(fs, hs, wl))

    def forward(self, x, y):
        sc_loss = 0.0
        mag_loss = 0.0
        
        for f in self.stft_losses:
            sc_l, mag_l = f(x, y)
            sc_loss += sc_l
            mag_loss += mag_l

        # Average over number of resolutions
        sc_loss /= len(self.stft_losses)
        mag_loss /= len(self.stft_losses)

        return sc_loss + mag_loss


class SpeechEnhancementLoss(nn.Module):
    """
    Final Hybrid Loss Function
    """
    def __init__(self, w_time=1.0, w_freq=1.0):
        super(SpeechEnhancementLoss, self).__init__()
        self.w_time = w_time
        self.w_freq = w_freq
        self.mr_stft_loss = MultiResolutionSTFTLoss()

    def forward(self, estimated_wav, clean_wav, estimated_spec=None, clean_spec=None):
        """
        Args:
            estimated_wav: The output waveform from the model [B, T]
            clean_wav: The ground truth clean waveform [B, T]
            estimated_spec: Optional direct spec output (not strictly needed for this loss)
        """
        
        # 1. Time Domain Loss (L1)
        # L1 is generally preferred over MSE for audio as it produces less "muddy" results
        time_loss = F.l1_loss(estimated_wav, clean_wav)

        # 2. Frequency Domain Loss (Multi-Res)
        # We re-compute STFT from the waveform to ensure consistency
        freq_loss = self.mr_stft_loss(estimated_wav, clean_wav)

        total_loss = (self.w_time * time_loss) + (self.w_freq * freq_loss)
        
        return total_loss, time_loss, freq_loss


In [8]:
import time
from pathlib import Path

import numpy as np
import torch
from torch.optim.lr_scheduler import StepLR

class BaseTrainer:
    def __init__(self,
                 config,
                 resume: bool,
                 model,
                 loss_function,
                 optimizer):
        self.n_gpu = torch.cuda.device_count()
        self.device = self._prepare_device(self.n_gpu, cudnn_deterministic=config["cudnn_deterministic"])

        self.optimizer = optimizer
        self.loss_function = loss_function

        self.model = model.to(self.device)

        if self.n_gpu > 1:
            self.model = torch.nn.DataParallel(self.model, device_ids=list(range(self.n_gpu)))

        # Trainer
        self.epochs = config["trainer"]["epochs"]
        self.save_checkpoint_interval = config["trainer"]["save_checkpoint_interval"]
        self.validation_config = config["trainer"]["validation"]
        self.validation_interval = self.validation_config["interval"]
        self.find_max = self.validation_config["find_max"]
        self.validation_custom_config = self.validation_config["custom"]

        # The following args is not in the config file. We will update it if the resume is True in later.
        self.start_epoch = 1
        self.best_score = -np.inf if self.find_max else np.inf
        self.root_dir = Path(config["root_dir"]).expanduser().absolute() / config["experiment_name"]
        self.checkpoints_dir = self.root_dir / "checkpoints"
        self.logs_dir = self.root_dir / "logs"
        prepare_empty_dir([self.checkpoints_dir, self.logs_dir], resume=resume)

        self.writer = writer(self.logs_dir.as_posix())


        if resume: self._resume_checkpoint()

        print("Configurations are as follows: ")


        self._print_networks([self.model])


    def _resume_checkpoint(self):
        """Resume experiment from the latest checkpoint.
        Notes:
            To be careful at the loading. if the model is an instance of DataParallel, we need to set model.module.*
        """
        latest_model_path = self.checkpoints_dir.expanduser().absolute() / "latest_model.tar"
        assert latest_model_path.exists(), f"{latest_model_path} does not exist, can not load latest checkpoint."

        checkpoint = torch.load(latest_model_path.as_posix(), map_location=self.device)

        self.start_epoch = checkpoint["epoch"] + 1
        self.best_score = checkpoint["best_score"]
        self.optimizer.load_state_dict(checkpoint["optimizer"])

        if isinstance(self.model, torch.nn.DataParallel):
            self.model.module.load_state_dict(checkpoint["model"])
        else:
            self.model.load_state_dict(checkpoint["model"])

        print(f"Model checkpoint loaded. Training will begin in {self.start_epoch} epoch.")

    def _save_checkpoint(self, epoch, is_best=False):
        """Save checkpoint to <root_dir>/checkpoints directory.
        It contains:
            - current epoch
            - best score in history
            - optimizer parameters
            - model parameters
        Args:
            is_best(bool): if current checkpoint got the best score, it also will be saved in <root_dir>/checkpoints/best_model.tar.
        """
        print(f"\t Saving {epoch} epoch model checkpoint...")

        # Construct checkpoint tar package
        state_dict = {
            "epoch": epoch,
            "best_score": self.best_score,
            "optimizer": self.optimizer.state_dict()
        }

        if isinstance(self.model, torch.nn.DataParallel):  # Parallel
            state_dict["model"] = self.model.module.cpu().state_dict()
        else:
            state_dict["model"] = self.model.cpu().state_dict()

        """
        Notes:
            - latest_model.tar:
                Contains all checkpoint information, including optimizer parameters, model parameters, etc. New checkpoint will overwrite old one.
            - model_<epoch>.pth:
                The parameters of the model. Follow-up we can specify epoch to inference.
            - best_model.tar:
                Like latest_model, but only saved when <is_best> is True.
        """
        torch.save(state_dict, (self.checkpoints_dir / "latest_model.tar").as_posix())
        # torch.save(state_dict["model"], (self.checkpoints_dir / f"model_{str(epoch).zfill(4)}.pth").as_posix())
        if is_best:
            print(f"\t Found best score in {epoch} epoch, saving...")
            torch.save(state_dict, (self.checkpoints_dir / "best_model.tar").as_posix())

        # Use model.cpu() or model.to("cpu") will migrate the model to CPU, at which point we need remigrate model back.
        # No matter tensor.cuda() or tensor.to("cuda"), if tensor in CPU, the tensor will not be migrated to GPU, but the model will.
        self.model.to(self.device)

    @staticmethod
    def _prepare_device(n_gpu: int, cudnn_deterministic=False):
        """Choose to use CPU or GPU depend on "n_gpu".
        Args:
            n_gpu(int): the number of GPUs used in the experiment.
                if n_gpu is 0, use CPU;
                if n_gpu > 1, use GPU.
            cudnn_deterministic (bool): repeatability
                cudnn.benchmark will find algorithms to optimize training. if we need to consider the repeatability of the experiment, set use_cudnn_deterministic to True
        """
        if n_gpu == 0:
            print("Using CPU in the experiment.")
            device = torch.device("cpu")
        else:
            if cudnn_deterministic:
                print(f"Using CuDNN deterministic mode with {n_gpu} gpu in the experiment.")
                torch.backends.cudnn.deterministic = True
                torch.backends.cudnn.benchmark = False
            print(f"Using CuDNN with {n_gpu} gpu in the experiment.")
            device = torch.device("cuda:0")

        return device

    def _is_best(self, score, find_max=True):
        """Check if the current model is the best.
        """
        if find_max and score >= self.best_score:
            self.best_score = score
            return True
        elif not find_max and score <= self.best_score:
            self.best_score = score
            return True
        else:
            return False

    @staticmethod
    def _transform_pesq_range(pesq_score):
        """transform PESQ range. From [-0.5 ~ 4.5] to [0 ~ 1].
        """
        return (pesq_score + 0.5) / 5

    @staticmethod
    def _print_networks(nets: list):
        print(f"This project contains {len(nets)} networks, the number of the parameters: ")
        params_of_all_networks = 0
        for i, net in enumerate(nets, start=1):
            params_of_network = 0
            for param in net.parameters():
                params_of_network += param.numel()

            print(f"\tNetwork {i}: {params_of_network / 1e6} million.")
            params_of_all_networks += params_of_network

        print(f"The amount of parameters is {params_of_all_networks / 1e6} million.")

    def _set_models_to_train_mode(self):
        self.model.train()

    def _set_models_to_eval_mode(self):
        self.model.eval()

    def train(self):
        for epoch in range(self.start_epoch, self.epochs + 1):
            print(f"============== {epoch} epoch ==============")
            print("[0 seconds] Begin training...")
            timer = ExecutionTime()

            self._set_models_to_train_mode()
            self._train_epoch(epoch)

            if self.save_checkpoint_interval != 0 and (epoch % self.save_checkpoint_interval == 0):
                self._save_checkpoint(epoch)

            if self.validation_interval != 0 and epoch % self.validation_interval == 0:
                print(f"[{timer.duration()} seconds] Training is over. Validation is in progress...")

                self._set_models_to_eval_mode()
                score = self._validation_epoch(epoch)

                if self._is_best(score, find_max=self.find_max):
                    self._save_checkpoint(epoch, is_best=True)

            print(f"[{timer.duration()} seconds] End this epoch.")

    def _train_epoch(self, epoch):
        raise NotImplementedError

    def _validation_epoch(self, epoch):
        raise NotImplementedError

In [9]:
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
import torch
import time
import os
from tqdm import tqdm


class Trainer(BaseTrainer):
    def __init__(
            self,
            config,
            resume: bool,
            model,
            loss_function,
            optimizer,
            train_dataloader,
            validation_dataloader,
    ):
        super(Trainer, self).__init__(config, resume, model, loss_function, optimizer)
        self.train_data_loader = train_dataloader
        self.validation_data_loader = validation_dataloader
        self.stft = ConvSTFT(512, 256, 512, 'hamming', 'complex').to(self.device)
        self.istft = ConviSTFT(512, 256, 512, 'hamming', 'complex').to(self.device)
        self.cpudevice = torch.device("cpu")
        self.criterion = SpeechEnhancementLoss().to(self.device)

    def _train_epoch(self, epoch):
        loss_total = 0.0
        num_batchs = len(self.train_data_loader)
        start_time = time.time()
        with tqdm(total = num_batchs) as pbar:
            for i, (mixture, clean, ref, name) in enumerate(self.train_data_loader):
                mixture = mixture.to(self.device)
                clean = clean.to(self.device)
                ref = ref.to(self.device)
                self.optimizer.zero_grad()

                enhanced_cpl, enhanced = self.model(mixture, ref)
                # loss = self.model.loss_snr(enhanced, clean)
                # loss = self.model.loss_pesq_snr_stoi(enhanced, clean, self.stoi_loss)
                if enhanced.dim() == 3:
                    enhanced = enhanced.squeeze(1)
                if clean.dim() == 3:
                    clean = clean.squeeze(1)
                # Compute Loss
                loss, time_comp, freq_comp = self.criterion(enhanced, clean)
                
                loss.backward()


                #print(loss)
                #print(amp_loss)
                #print(phase_loss)
                self.optimizer.step()

                loss_total += loss.item()
                pbar.update(1)
            end_time = time.time()
            dl_len = len(self.train_data_loader)
            print("loss:", loss_total / dl_len)
            self.writer.add_scalar(f"Train/Loss", loss_total / dl_len, epoch)

    @torch.inference_mode()
    def _validation_epoch(self, epoch):
        loss_total = 0.0
        visualize_audio_limit = self.validation_custom_config["visualize_audio_limit"]
        visualize_waveform_limit = self.validation_custom_config["visualize_waveform_limit"]
        visualize_spectrogram_limit = self.validation_custom_config["visualize_spectrogram_limit"]
        num_batchs = len(self.validation_data_loader)
        sample_length = self.validation_custom_config["sample_length"]
        self.stft = ConvSTFT(512, 256, 512, 'hamming', 'complex').to(self.device)
        self.istft = ConviSTFT(400, 100, 512, 'hamming', 'complex').to(self.device)
        stoi_c_n = []  # clean and noisy
        stoi_c_e = []  # clean and enhanced
        pesq_c_n = []
        pesq_c_e = []
        ssnr_c_n = []
        ssnr_c_e = []
        Lpesq_mos = []
        LCSIG = []
        LCBAK = []
        LCOVL = []
        LsegSNR = []
        LSTOI = []
        with tqdm(total = num_batchs) as pbar:
            for i, (mixture1, clean1, ref1, mixture2, clean2, ref2, name) in enumerate(self.validation_data_loader):
                assert len(name) == 1, "Only support batch size is 1 in enhancement stage."
                name = name[0]
                padded_length = 0

                mixture1 = mixture1.to(self.device)  # [1, 1, T]
                mixture2 = mixture2.to(self.device)
                clean1 = clean1.to(self.device)
                clean2 = clean2.to(self.device)
                ref1 = ref1.to(self.device)
                ref2 = ref2.to(self.device)


                enhanced_cpl1, enhanced1 = self.model(mixture1, ref1)
                enhanced_cpl2, enhanced2 = self.model(mixture2, ref2)

                
                mixture = torch.cat((mixture1, mixture2), dim=2)
                clean = torch.cat((clean1, clean2), dim=2)
                enhanced = torch.cat((enhanced1, enhanced2), dim=1)

                #print(enhanced_cpl)
                loss = self.model.loss_snr(enhanced, clean)
                # loss = self.model.loss_pesq_snr_stoi(enhanced, clean, self.stoi_loss)
                loss_total += loss.item()

                #enhanced = self.istft(enhanced_cpl, None)


                enhanced = enhanced.reshape(-1).cpu().numpy()
                clean = clean.cpu().numpy().reshape(-1)
                mixture = mixture.cpu().numpy().reshape(-1)
                assert len(mixture) == len(enhanced) == len(clean)

                pesq_mos, CSIG, CBAK, COVL, segSNR, STOI = compute_metrics(clean, enhanced,16000, 0)
                stoi_c_n.append(compute_STOI(clean, mixture, sr=16000))
                stoi_c_e.append(compute_STOI(clean, enhanced, sr=16000))
                pesq_c_n.append(compute_PESQ(clean, mixture, sr=16000))
                pesq_c_e.append(compute_PESQ(clean, enhanced, sr=16000))
                ssnr_c_n.append(si_snr(mixture, clean).item())
                ssnr_c_e.append(si_snr(enhanced, clean).item())
                Lpesq_mos.append(pesq_mos)
                LCSIG.append(CSIG)
                LCBAK.append(CBAK)
                LCOVL.append(COVL)
                LsegSNR.append(segSNR)
                LSTOI.append(STOI)

            # Visualize audio
                if i <= visualize_audio_limit:
                    self.writer.add_audio(f"Speech/{name}_Noisy", mixture, epoch, sample_rate=16000)
                    self.writer.add_audio(f"Speech/{name}_Enhanced", enhanced, epoch, sample_rate=16000)
                    self.writer.add_audio(f"Speech/{name}_Clean", clean, epoch, sample_rate=16000)

                

                pbar.update(1)
            # Metric


        get_metrics_ave = lambda metrics: np.sum(metrics) / len(metrics)

        print(f"STOI Clean and noisy : {get_metrics_ave(stoi_c_n)}")
        print(f"STOI Clean and enhanced : {get_metrics_ave(stoi_c_e)}")
        print(f"PESQ Clean and noisy : {get_metrics_ave(pesq_c_n)}")
        print(f"PESQ Clean and enhanced : {get_metrics_ave(pesq_c_e)}")
        print(f"SI-SNR Clean and noisy : {get_metrics_ave(ssnr_c_n)}")
        print(f"SI-SNR Clean and enhanced : {get_metrics_ave(ssnr_c_e)}")
        print(f"Lpesq_mos Clean and enhanced : {get_metrics_ave(Lpesq_mos)}")
        print(f"LCSIG Clean and enhanced : {get_metrics_ave(LCSIG)}")
        print(f"LCBAK Clean and enhanced : {get_metrics_ave(LCBAK)}")
        print(f"LCOVL Clean and enhanced : {get_metrics_ave(LCOVL)}")
        print(f"LsegSNR Clean and enhanced : {get_metrics_ave(LsegSNR)}")
        print(f"LSTOI Clean and enhanced : {get_metrics_ave(LSTOI)}")

        dl_len = len(self.validation_data_loader)
        print("loss:", loss_total / dl_len)
        score = loss_total
        return score


In [ ]:
import argparse
import numpy as np
import torch
from torch.utils.data import DataLoader

# Define the JSON content as a dictionary
config = {
    "seed": 0,
    "description": "descript",
    "root_dir": "/home/jovyan/pentaSearch/WaveUNet",
    "cudnn_deterministic": False,
    "trainer": {
        "epochs": 30, # TODO
        "save_checkpoint_interval": 1,
        "validation": {
            "interval": 1,
            "find_max": False,
            "custom": {
                "visualize_audio_limit": 1,
                "visualize_waveform_limit": 1,
                "visualize_spectrogram_limit": 1,
                "sample_length": 64000
            }
        }
    },


}

def main(config, resume):
    # Set seeds
    torch.manual_seed(config["seed"])  # for both CPU and GPU
    np.random.seed(config["seed"])
    
    # Initialize datasets
    train_dataset = Dataset(
        dataset="/home/jovyan/file_paths_combined.txt",
        limit=11572,
        offset=0,
        sample_length=32000,
        mode="train"
    )
    validation_dataset = Dataset(
        dataset="/home/jovyan/file_paths_combined.txt",
        limit=12396,
        offset=11572,
        sample_length=32000,
        mode="validation"
    )
    # Initialize dataloaders
    train_dataloader = DataLoader(
        dataset=train_dataset,
        batch_size=2,
        num_workers=12,
        shuffle=True,
        pin_memory=True,
        persistent_workers=True,
        prefetch_factor=4
    )
    valid_dataloader = DataLoader(
        dataset=validation_dataset,
        num_workers=6,
        batch_size=1,
        pin_memory=True,
        
    )

    # Initialize model directly
    model = Model()  # No parameters to pass in this example


    # Initialize optimizer directly
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4) # TODO
    # optimizer = torch.optim.Adam(
    #     params=model.parameters(),
    #     lr=0.001,
    #     betas=(0.9, 0.999)
    # )

    # Initialize loss function directly
    loss_function = mse_loss()  # No parameters to pass in this example

    # Initialize trainer directly
    trainer = Trainer(
        config=config,
        resume=resume,
        model=model,
        loss_function=loss_function,
        optimizer=optimizer,
        train_dataloader=train_dataloader,
        validation_dataloader=valid_dataloader,
    )

    # Start training
    trainer.train()

if __name__ == '__main__':
    config["experiment_name"] = 'config'
    config["config_path"] = 'config.json'
    main(config, False) # TODO


Using CuDNN with 1 gpu in the experiment.
Configurations are as follows: 
This project contains 1 networks, the number of the parameters: 
	Network 1: 4.031388 million.
The amount of parameters is 4.031388 million.
============== 1 epoch ==============
[0 seconds] Begin training...


  2% 138/5786 [00:34<16:52,  5.58it/s] 